# Endotoxin assay plate setup

This notebook prepares the 0.005-5 EU/mL endotoxin standards and loads 30 wells of a black Corning 360 µL flat-bottom plate. It follows the calibrated Kimble 10 × 75 mm tube geometry from `kimble_4mL_tube_geometry_test.ipynb`.

> **Intentional protocol deviation:** working solution is dispensed before samples, standards, and EFW, as requested. The kit guide instead recommends adding samples/standards, preheating for 10 minutes at 37 °C, and adding working solution last. Validate this deviation before using assay results.

Run cells from top to bottom once. Confirm the printed deck and volume plan before setup. Do not rerun deck construction or a partially completed stage. All tips and direct-contact labware must be endotoxin-free.


In [1]:
%load_ext autoreload
%autoreload 2

import asyncio
from collections import defaultdict
from copy import deepcopy

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend, LiquidHandlerChatterboxBackend
from pylabrobot.liquid_handling.standard import Mix
from pylabrobot.resources import (
    Kimble_tube_4mL_Rb,
    cor_96_wellplate_360uL_Fb,
    hamilton_96_tiprack_1000uL_filter,
    hamilton_96_tiprack_300uL_filter,
    hamilton_96_tiprack_50uL_filter,
    set_volume_tracking,
)
from pylabrobot.resources.coordinate import Coordinate
from pylabrobot.resources.hamilton import (
    STARLetDeck,
    TIP_CAR_480_A00,
    hamilton_mfx_carrier_L5_base,
    hamilton_tube_carrier_32_a00_insert_eppendorf_1_5mL,
)
from pylabrobot.resources.hamilton.mfx_modules import (
    hamilton_mfx_plateholder_DWP_metal_tapped,
)


## Configuration and protocol plan

Set `SIMULATE = True` only for a no-hardware dry run. PyLabRobot channel indices are zero-based; physical channel 1 is index 0 and physical channels 5-6 are indices 4-5. Physical tube position 32 is the frontmost carrier site and maps to PyLabRobot site 0.


In [2]:
SIMULATE = False
SINGLE_CHANNEL = 0
WORKING_CHANNELS = [4, 5]
KIMBLE_X_OFFSET_MM = -1.75
KIMBLE_Y_OFFSET_MM = 1.5
KIMBLE_SEATING_Z_OFFSET_MM = 15.0
KIMBLE_LLD_SEARCH_HEIGHT_MM = 75.0  # Relative to the tube's inner bottom; ~1 mm above its rim.
KIMBLE_INNER_BOTTOM_MM = 5.0
TUBE_ASPIRATE_HEIGHT_MM = 6.0
TUBE_DISPENSE_HEIGHT_MM = 12.0
STANDARD_MIX_VOLUME_UL = 900.0
STANDARD_MIX_REPETITIONS = 2
STANDARD_MIX_FLOW_UL_S = 400.0
# STANDARD_MIX_CYCLES = 4
STANDARD_MIX_CYCLES = 1
STANDARD_PAUSE_SECONDS = 0.0 if SIMULATE else 20.0
set_volume_tracking(enabled=True)

def physical_to_site(position: int) -> int:
    if not 1 <= position <= 32:
        raise ValueError(f"Physical tube position must be 1-32, got {position}.")
    return 32 - position

RAIL19_TUBES = {
    "efw_standard": {"position": 32, "initial_ul": 4000.0},
    "efw_plate": {"position": 31, "initial_ul": 4000.0},
    "efw_reserve_1": {"position": 30, "initial_ul": 4000.0},
    "efw_reserve_2": {"position": 29, "initial_ul": 4000.0},
    "std_0_005": {"position": 28, "initial_ul": 0.0, "eu_ml": 0.005},
    "std_0_05": {"position": 27, "initial_ul": 0.0, "eu_ml": 0.05},
    "std_0_5": {"position": 26, "initial_ul": 0.0, "eu_ml": 0.5},
    "std_5": {"position": 25, "initial_ul": 0.0, "eu_ml": 5.0},
    "working_1": {"position": 23, "initial_ul": 2000.0},
    "working_2": {"position": 22, "initial_ul": 2000.0},
}
RAIL18_TUBES = {
    "sample_1": {"position": 32, "initial_ul": 1000.0},
    "sample_2": {"position": 31, "initial_ul": 1000.0},
}
STANDARD_PREP = [
    {"key": "std_5", "efw_ul": 750.0, "stock_ul": 250.0, "mix_tip": "B1"},
    {"key": "std_0_5", "efw_ul": 900.0, "from": "std_5", "transfer_ul": 100.0, "transfer_tip": "F2", "mix_tip": "C1"},
    {"key": "std_0_05", "efw_ul": 900.0, "from": "std_0_5", "transfer_ul": 100.0, "transfer_tip": "G2", "mix_tip": "D1"},
    {"key": "std_0_005", "efw_ul": 900.0, "from": "std_0_05", "transfer_ul": 100.0, "transfer_tip": "H2", "mix_tip": "E1"},
]
WORKING_WELLS = [f"A{i}" for i in range(1, 13)] + [f"B{i}" for i in range(1, 13)] + ["C1", "C2", "D1", "D2", "E1", "E2"]

PLATE_TRANSFER_GROUPS = [
    {"source": "efw_plate", "tip_size": 300, "tip": "F1", "destinations": [
        ("A1", 100.0), ("A2", 100.0),
        ("A5", 50.0), ("A6", 50.0), ("B5", 50.0), ("B6", 50.0),
        ("A7", 75.0), ("A8", 75.0), ("B7", 75.0), ("B8", 75.0),
        ("A9", 87.5), ("B9", 87.5), ("A10", 93.75), ("B10", 93.75),
    ]},
    {"source": "sample_1", "tip_size": 300, "tip": "G1", "destinations": [("A3", 100.0), ("A4", 100.0), ("A5", 50.0), ("A6", 50.0), ("A11", 50.0), ("A12", 50.0)]},
    {"source": "sample_2", "tip_size": 300, "tip": "H1", "destinations": [("B3", 100.0), ("B4", 100.0), ("B5", 50.0), ("B6", 50.0), ("B11", 50.0), ("B12", 50.0)]},
    {"source": "std_0_005", "tip_size": 300, "tip": "A2", "destinations": [("B1", 100.0), ("B2", 100.0)]},
    {"source": "std_0_05", "tip_size": 300, "tip": "B2", "destinations": [("C1", 100.0), ("C2", 100.0)]},
    {"source": "std_0_5", "tip_size": 300, "tip": "C2", "destinations": [("D1", 100.0), ("D2", 100.0), ("A11", 50.0), ("A12", 50.0), ("B11", 50.0), ("B12", 50.0)]},
    {"source": "std_5", "tip_size": 300, "tip": "D2", "destinations": [("E1", 100.0), ("E2", 100.0)]},
    {"source": "sample_1", "tip_size": 50, "tip": "A1", "destinations": [("A7", 25.0), ("A8", 25.0), ("A9", 12.5), ("A10", 6.25)]},
    {"source": "sample_2", "tip_size": 50, "tip": "B1", "destinations": [("B7", 25.0), ("B8", 25.0), ("B9", 12.5), ("B10", 6.25)]},
]
PLATE_LABELS = {
    "A1": "Blank", "A2": "Blank", "A3": "sample 1 1:1", "A4": "sample 1 1:1",
    "A5": "sample 1 1:2", "A6": "sample 1 1:2", "A7": "sample 1 1:4", "A8": "sample 1 1:4",
    "A9": "sample 1 1:8", "A10": "sample 1 1:16", "A11": "sample 1 SPIKED 1:2", "A12": "sample 1 SPIKED 1:2",
    "B1": "Std1 0.005 EU/mL", "B2": "Std1 0.005 EU/mL", "B3": "sample 2 1:1", "B4": "sample 2 1:1",
    "B5": "sample 2 1:2", "B6": "sample 2 1:2", "B7": "sample 2 1:4", "B8": "sample 2 1:4",
    "B9": "sample 2 1:8", "B10": "sample 2 1:16", "B11": "sample 2 SPIKED 1:2", "B12": "sample 2 SPIKED 1:2",
    "C1": "Std2 0.05 EU/mL", "C2": "Std2 0.05 EU/mL", "D1": "Std3 0.5 EU/mL", "D2": "Std3 0.5 EU/mL",
    "E1": "Std4 5 EU/mL", "E2": "Std4 5 EU/mL",
}


## Pure preflight validation

This validation performs no motion. It verifies the corrected plate map, source residuals, tip policy, serial dilution routing, and carrier position conversion.


In [3]:
def required_tip_size(volume_ul: float) -> int:
    if 50.0 <= volume_ul <= 100.0:
        return 300
    if 0.0 < volume_ul < 50.0:
        return 50
    raise ValueError(f"Plate transfer volume is outside the supported range: {volume_ul} µL")

def validate_protocol_plan():
    assert physical_to_site(32) == 0 and physical_to_site(31) == 1
    assert physical_to_site(28) == 4 and physical_to_site(22) == 10
    assert 24 not in {cfg["position"] for cfg in RAIL19_TUBES.values()}
    assert len(RAIL19_TUBES) == 10 and len(RAIL18_TUBES) == 2
    assert [RAIL19_TUBES[s["key"]]["eu_ml"] for s in STANDARD_PREP] == [5.0, 0.5, 0.05, 0.005]
    assert all(s["efw_ul"] == 900.0 for s in STANDARD_PREP[1:])
    assert STANDARD_PREP[0]["efw_ul"] == 750.0 and STANDARD_PREP[0]["stock_ul"] == 250.0

    plate_additions = defaultdict(float)
    source_use = defaultdict(float)
    for group in PLATE_TRANSFER_GROUPS:
        for well_name, volume_ul in group["destinations"]:
            assert required_tip_size(volume_ul) == group["tip_size"], (group, well_name, volume_ul)
            plate_additions[well_name] += volume_ul
            source_use[group["source"]] += volume_ul
    assert set(WORKING_WELLS) == set(PLATE_LABELS) == set(plate_additions)
    assert len(WORKING_WELLS) == 30 and len(set(WORKING_WELLS)) == 30
    assert all(abs(plate_additions[w] - 100.0) < 1e-9 for w in WORKING_WELLS)
    assert all(100.0 + plate_additions[w] == 200.0 for w in WORKING_WELLS)
    assert source_use["efw_plate"] == 1062.5
    assert source_use["sample_1"] == source_use["sample_2"] == 468.75
    assert source_use["std_0_5"] == 400.0
    assert dict(PLATE_TRANSFER_GROUPS[5])["source"] == "std_0_5"

    expected_residuals = {
        "efw_standard": 550.0, "efw_plate": 2937.5,
        "efw_reserve_1": 4000.0, "efw_reserve_2": 4000.0,
        "working_1": 500.0, "working_2": 500.0,
        "sample_1": 531.25, "sample_2": 531.25,
        "std_5": 700.0, "std_0_5": 500.0,
        "std_0_05": 700.0, "std_0_005": 800.0,
    }
    assert max(200.0 for _ in WORKING_WELLS) <= 360.0
    return {
        "plate_final_volumes_ul": {w: 200.0 for w in WORKING_WELLS},
        "plate_labels": deepcopy(PLATE_LABELS),
        "source_use_ul": dict(source_use),
        "expected_residuals_ul": expected_residuals,
    }

PLAN_CHECK = validate_protocol_plan()
print("Protocol plan validated. Expected source residuals (µL):")
for source, volume in PLAN_CHECK["expected_residuals_ul"].items():
    print(f"  {source:16s} {volume:8.2f}")


Protocol plan validated. Expected source residuals (µL):
  efw_standard       550.00
  efw_plate         2937.50
  efw_reserve_1     4000.00
  efw_reserve_2     4000.00
  working_1          500.00
  working_2          500.00
  sample_1           531.25
  sample_2           531.25
  std_5              700.00
  std_0_5            500.00
  std_0_05           700.00
  std_0_005          800.00


## Deck construction

The installed function `hamilton_tube_carrier_32_a00_insert_eppendorf_1_5mL` documents its Hamilton name as `SMP_CAR_32_A00`. Five catalog 188042 DWP modules are installed on the catalog 188039 carrier. Only module 0 contains a plate.


In [4]:
backend = LiquidHandlerChatterboxBackend(num_channels=8) if SIMULATE else STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())

tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
tips_1000 = hamilton_96_tiprack_1000uL_filter(name="tips_1000ul_filter")
tips_300 = hamilton_96_tiprack_300uL_filter(name="tips_300ul_filter")
tips_50 = hamilton_96_tiprack_50uL_filter(name="tips_50ul_filter")
tip_carrier[0].assign_child_resource(tips_1000)
tip_carrier[1].assign_child_resource(tips_300)
tip_carrier[2].assign_child_resource(tips_50)
lh.deck.assign_child_resource(tip_carrier, rails=25)

tube_carrier_19 = hamilton_tube_carrier_32_a00_insert_eppendorf_1_5mL(name="smp_car_32_rail19")
tube_carrier_18 = hamilton_tube_carrier_32_a00_insert_eppendorf_1_5mL(name="smp_car_32_rail18")
tube_resources = {}

def assign_kimble(
    carrier, rail: int, key: str, position: int, initial_ul: float,
    x_offset_mm: float = KIMBLE_X_OFFSET_MM, y_offset_mm: float = KIMBLE_Y_OFFSET_MM,
):
    tube = Kimble_tube_4mL_Rb(name=f"rail{rail}_{key}_position{position}")
    holder = carrier[physical_to_site(position)]
    location = Coordinate(
        (holder.get_size_x() - tube.get_size_x()) / 2 + x_offset_mm,
        (holder.get_size_y() - tube.get_size_y()) / 2 + y_offset_mm,
        KIMBLE_SEATING_Z_OFFSET_MM,
    )
    holder.assign_child_resource(tube, location=location)
    tube.set_volume(initial_ul)
    tube_resources[key] = tube
    return tube

for key, cfg in RAIL19_TUBES.items():
    assign_kimble(tube_carrier_19, 19, key, cfg["position"], cfg["initial_ul"])
for key, cfg in RAIL18_TUBES.items():
    assign_kimble(tube_carrier_18, 18, key, cfg["position"], cfg["initial_ul"])
lh.deck.assign_child_resource(tube_carrier_18, rails=18)
lh.deck.assign_child_resource(tube_carrier_19, rails=19)

mfx_modules = {i: hamilton_mfx_plateholder_DWP_metal_tapped(name=f"rail7_dwp_module_{i}") for i in range(5)}
mfx_carrier = hamilton_mfx_carrier_L5_base(name="mfx_carrier_188039_rail7", modules=mfx_modules)
assay_plate = cor_96_wellplate_360uL_Fb(name="black_endotoxin_assay_plate")
mfx_modules[0].assign_child_resource(assay_plate)
lh.deck.assign_child_resource(mfx_carrier, rails=7)

assert tube_carrier_19.model == "hamilton_tube_carrier_32_a00_insert_eppendorf_1_5mL"
assert tube_carrier_19[0].children[0] is tube_resources["efw_standard"]
assert tube_carrier_19[physical_to_site(24)].children == []
assert all(tube.get_size_x() == 10 and tube.get_size_y() == 10 and tube.get_size_z() == 75 for tube in tube_resources.values())
assert tips_1000["A1"][0].get_tip().nominal_volume == 1000 and tips_1000["A1"][0].get_tip().has_filter
assert tips_300["A1"][0].get_tip().nominal_volume == 300 and tips_300["A1"][0].get_tip().has_filter
assert tips_50["A1"][0].get_tip().nominal_volume == 50 and tips_50["A1"][0].get_tip().has_filter
assert assay_plate.get_item("A1").max_volume == 360

print(lh.deck.summary())


Rail  Resource                               Type                 Coordinates (mm)
(-6)  ├── trash_core96                       Trash                (-58.200, 106.000, 216.400)
      │
(7)   ├── mfx_carrier_188039_rail7           MFXCarrier           (235.000, 063.000, 100.000)
      │   ├── black_endotoxin_assay_plate    Plate                (239.000, 072.000, 180.920)
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │
(18)  ├── smp_car_32_rail18                  TubeCarrier          (482.500, 063.000, 100.000)
      │   ├── rail18_sample_1_position32     Tube                 (490.250, 074.000, 140.880)
      │   ├── rail18_sample_2_position31     Tube                 (490.250, 089.000, 140.880)
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │

## Connect and define execution helpers

Confirm the physical deck exactly matches the printed summary. Standard preparation consumes EFW from position 32; plate dilution uses EFW from position 31. Positions 30 and 29 remain reserves.


In [5]:
if not lh.setup_finished:
    if SIMULATE:
        await lh.setup()
    else:
        await lh.setup(skip_autoload=True)

protocol_state = {"standards_prepared": False, "working_dispensed": False, "plate_loaded": False}
attached_channels = set()

async def confirm(message: str):
    if SIMULATE:
        print(f"[SIMULATION CONFIRMED] {message}")
        return
    response = input(f"\n{message} Type GO and press Enter to continue: " ).strip().upper()
    if response != "GO":
        raise RuntimeError("Operator cancelled the protocol.")

def plate_well(name: str):
    return assay_plate.get_item(name)

def plate_dispense_height(name: str) -> float:
    well = plate_well(name)
    current_ul = well.tracker.get_used_volume()
    if current_ul <= 0:
        return 1.0
    return min(well.compute_height_from_volume(current_ul) + 1.0, well.get_size_z() - 1.0)

async def pickup(tip_rack, positions, channels):
    spots = [tip_rack.get_item(position) for position in positions]
    await lh.pick_up_tips(spots, use_channels=channels)
    attached_channels.update(channels)

async def discard(channels):
    await lh.discard_tips(use_channels=channels)
    attached_channels.difference_update(channels)

def kimble_aspiration_geometry(tube, aspiration_ul: float):
    """Return explicit surface height and following distance for a Kimble 4 mL tube."""
    current_ul = tube.tracker.get_used_volume()
    if current_ul < aspiration_ul:
        raise RuntimeError(f"{tube.name} has {current_ul:.1f} µL tracked; cannot aspirate {aspiration_ul:.1f} µL.")
    current_height_mm = tube.compute_height_from_volume(current_ul)
    final_height_mm = tube.compute_height_from_volume(current_ul - aspiration_ul)
    surface_following_mm = current_height_mm - final_height_mm
    return current_ul, current_height_mm, surface_following_mm

async def add_efw_to_standard_tubes():
    await pickup(tips_1000, ["A1"], [SINGLE_CHANNEL])
    efw_tube = tube_resources["efw_standard"]
    for step in STANDARD_PREP:
        volume_ul = step["efw_ul"]
        current_ul, liquid_height_mm, surface_following_mm = kimble_aspiration_geometry(efw_tube, volume_ul)
        print(
            f"EFW standard: aspirating {volume_ul:.1f} µL from {current_ul:.1f} µL; "
            f"liquid_height={liquid_height_mm:.2f} mm, surface_following={surface_following_mm:.2f} mm."
        )
        await lh.aspirate(
            [efw_tube], 
            vols=[volume_ul],
            use_channels=[SINGLE_CHANNEL],
            liquid_height=[liquid_height_mm],
            surface_following_distance=[surface_following_mm],
            lld_mode=[STARBackend.LLDMode.GAMMA],
            lld_search_height=[KIMBLE_LLD_SEARCH_HEIGHT_MM],
            immersion_depth=[2.0],
        )
        await lh.dispense(
            [tube_resources[step["key"]]],
            vols=[volume_ul],
            use_channels=[SINGLE_CHANNEL],
            liquid_height=[TUBE_DISPENSE_HEIGHT_MM],
            blow_out=[True],
            settling_time=[1])
    await discard([SINGLE_CHANNEL])

async def mix_standard(standard_key: str, tip_position: str):
    await pickup(tips_1000, [tip_position], [SINGLE_CHANNEL])
    for cycle in range(1, STANDARD_MIX_CYCLES + 1):
        await lh.aspirate([tube_resources[standard_key]], vols=[STANDARD_MIX_VOLUME_UL], use_channels=[SINGLE_CHANNEL], liquid_height=[TUBE_ASPIRATE_HEIGHT_MM], flow_rates=[STANDARD_MIX_FLOW_UL_S])
        await lh.dispense(
            [tube_resources[standard_key]], vols=[STANDARD_MIX_VOLUME_UL], use_channels=[SINGLE_CHANNEL],
            liquid_height=[6.0], flow_rates=[400],
            mix=[Mix(volume=STANDARD_MIX_VOLUME_UL, repetitions=STANDARD_MIX_REPETITIONS, flow_rate=STANDARD_MIX_FLOW_UL_S)],
            blow_out=[True], settling_time=[1],
        )
        print(f"{standard_key}: completed mix/pause cycle {cycle}/{STANDARD_MIX_CYCLES}.")
        await asyncio.sleep(STANDARD_PAUSE_SECONDS)
    await discard([SINGLE_CHANNEL])

async def serial_transfer(source_key: str, destination_key: str, tip_position: str):
    await pickup(tips_1000, [tip_position], [SINGLE_CHANNEL])
    await lh.aspirate([tube_resources[source_key]], vols=[100.0], use_channels=[SINGLE_CHANNEL], liquid_height=[TUBE_ASPIRATE_HEIGHT_MM])
    await lh.dispense([tube_resources[destination_key]], vols=[100.0], use_channels=[SINGLE_CHANNEL], liquid_height=[TUBE_DISPENSE_HEIGHT_MM], blow_out=[True], settling_time=[1])
    await discard([SINGLE_CHANNEL])

async def dispense_group(group):
    rack = tips_300 if group["tip_size"] == 300 else tips_50
    await pickup(rack, [group["tip"]], [SINGLE_CHANNEL])
    source = tube_resources[group["source"]]
    for destination, volume_ul in group["destinations"]:
        await lh.aspirate([source], vols=[volume_ul], use_channels=[SINGLE_CHANNEL], liquid_height=[TUBE_ASPIRATE_HEIGHT_MM])
        await lh.dispense([plate_well(destination)], vols=[volume_ul], use_channels=[SINGLE_CHANNEL], liquid_height=[plate_dispense_height(destination)], blow_out=[True], settling_time=[1])
    await discard([SINGLE_CHANNEL])

async def recover_attached_tips():
    if not attached_channels:
        print("No tips are recorded as attached.")
        return
    channels = sorted(attached_channels)
    await confirm(f"Recovery will discard tips from physical channels {[c + 1 for c in channels]}. Verify it is safe to move.")
    await discard(channels)
    print("Attached-tip recovery complete.")


2026-07-08 09:54:12,957 - pylabrobot.io.usb - INFO - Finding USB device...
2026-07-08 09:54:12,967 - pylabrobot.io.usb - INFO - Found USB device.
2026-07-08 09:54:12,971 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-07-08 09:54:16,135 - pylabrobot - INFO - Running backend initialization procedure.


In [6]:
# await pickup(tips_1000, ["A1"], channels=[0])

In [7]:

# await lh.aspirate(
#     [tube_resources["efw_standard"]],
#     vols=[900],
#     use_channels=[0],
#     liquid_height = [56.1],
#     # lld_mode=[STARBackend.LLDMode.GAMMA],
#     # auto_surface_following_distance=[True],
#     # probe_liquid_height=[True],
#     # lld_search_height=[KIMBLE_LLD_SEARCH_HEIGHT_MM],
#     # immersion_depth=[2.0],
# )

In [8]:
# await lh.dispense(
#     [tube_resources["efw_standard"]],
#     vols=[900],
#     use_channels=[0],
#     liquid_height = [55.1],
#     swap_speed = [160],
#     # lld_mode=[STARBackend.LLDMode.GAMMA],
#     # auto_surface_following_distance=[True],
#     # probe_liquid_height=[False],
#     # lld_search_height=[KIMBLE_LLD_SEARCH_HEIGHT_MM],
#     # immersion_depth=[2.0],
# )

## Protocol stages

The stages are independently callable but enforce their required order. `run_protocol()` performs all three. If a stage fails after motion begins, do not rerun it blindly; inspect the physical deck and use the recovery cell if a tip remains attached.


In [ ]:
async def prepare_standards():
    if protocol_state["standards_prepared"]:
        raise RuntimeError("Standards are already marked prepared.")
    await confirm("Verify rail 19 positions 25-28 are empty standard tubes, position 32 contains 4.0 mL EFW, and all specified filtered tips are present.")
    await add_efw_to_standard_tubes()
    await confirm("Add exactly 250 µL of resolubilized 20 EU/mL endotoxin stock to physical position 25 (the tube containing 750 µL EFW).")
    tube_resources["std_5"].set_volume(1000.0)
    await mix_standard("std_5", STANDARD_PREP[0]["mix_tip"])
    for step in STANDARD_PREP[1:]:
        await serial_transfer(step["from"], step["key"], step["transfer_tip"])
        await mix_standard(step["key"], step["mix_tip"])
    protocol_state["standards_prepared"] = True
    print("Standards prepared: positions 25-28 contain 5, 0.5, 0.05, and 0.005 EU/mL, respectively.")

async def dispense_working_solution():
    # if not protocol_state["standards_prepared"]:
    #     raise RuntimeError("Prepare standards before dispensing working solution.")
    if protocol_state["working_dispensed"]:
        raise RuntimeError("Working solution is already marked dispensed.")
    await confirm("Verify 2 mL fresh working solution is present in rail 19 physical positions 23 and 22. Working solution will be added first, contrary to the kit guide.")
    await pickup(tips_1000, ["F1", "G1"], WORKING_CHANNELS)
    sources = [tube_resources["working_1"], tube_resources["working_2"]]
    channel_wells = [
        [f"A{column}" for column in range(1, 13)] + ["C1", "D1", "E1"],
        [f"B{column}" for column in range(1, 13)] + ["C2", "D2", "E2"],
    ]
    batch_size = 6
    dispense_volume_ul = 100.0 # I agree this should be 100 but errs; 70 works. 
    return_volume_ul = 100.0
    for start_index in range(0, len(channel_wells[0]), batch_size):
        batches = [wells[start_index:start_index + batch_size] for wells in channel_wells]
        aspirate_volumes = [len(batch) * dispense_volume_ul + return_volume_ul for batch in batches]
        aspiration_geometries = [
            kimble_aspiration_geometry(source, volume_ul)
            for source, volume_ul in zip(sources, aspirate_volumes)
        ]
        source_heights = [geometry[1] for geometry in aspiration_geometries]
        surface_following_distances = [geometry[2] for geometry in aspiration_geometries]
        print(
            f"Working solution batch {start_index // batch_size + 1}: "
            f"aspirating {aspirate_volumes[0]:.1f} µL per source at "
            f"heights {[round(height, 2) for height in source_heights]} mm."
        )
        await lh.aspirate(
            sources,
            vols=aspirate_volumes,
            use_channels=WORKING_CHANNELS,
            liquid_height=source_heights,
            surface_following_distance=surface_following_distances,
            transport_air_volume=[0,0]
            )
        for dispense_index, destination_names in enumerate(zip(*batches)):
            destinations = [plate_well(name) for name in destination_names]
            await lh.dispense(
                destinations,
                vols=[100.0, 100.0],
                use_channels=WORKING_CHANNELS,
                liquid_height=[4.0, 4.0],
                blow_out=[False, False],
                settling_time=[1, 1],
                transport_air_volume=[0,0],
            )
        # Return up to the extra 100 µL reserve to the working-solution source tubes.
        pending_return_indices = list(range(len(WORKING_CHANNELS)))
        attempted_return_ul = return_volume_ul
        returned_by_channel = {}
        last_return_error = None
        while pending_return_indices and attempted_return_ul > 0:
            attempt_sources = [sources[index] for index in pending_return_indices]
            attempt_channels = [WORKING_CHANNELS[index] for index in pending_return_indices]
            attempt_heights = [source.compute_height_from_volume(source.tracker.get_used_volume()) for source in attempt_sources]
            try:
                await lh.dispense(
                    attempt_sources,
                    vols=[attempted_return_ul] * len(attempt_sources),
                    use_channels=attempt_channels,
                    liquid_height=attempt_heights,
                    transport_air_volume=[0] * len(attempt_sources),
                    blow_out=[True] * len(attempt_sources),
                )
            except Exception as error:
                last_return_error = error
                channel_errors = getattr(error, "errors", None)
                if channel_errors is not None:
                    failed_channels = set(channel_errors)
                    succeeded_indices = [index for index in pending_return_indices if WORKING_CHANNELS[index] not in failed_channels]
                    for index in succeeded_indices:
                        returned_by_channel[WORKING_CHANNELS[index]] = attempted_return_ul
                    pending_return_indices = [index for index in pending_return_indices if WORKING_CHANNELS[index] in failed_channels]
                attempted_return_ul -= 5.0
                if pending_return_indices:
                    print(
                        f"Return-to-source dispense failed for channels "
                        f"{[channel + 1 for channel in [WORKING_CHANNELS[index] for index in pending_return_indices]]}; "
                        f"retrying with {attempted_return_ul:.1f} µL. Error: {error}"
                    )
            else:
                for index in pending_return_indices:
                    returned_by_channel[WORKING_CHANNELS[index]] = attempted_return_ul
                pending_return_indices = []
        if pending_return_indices:
            raise RuntimeError("Could not return working-solution reserve to source tubes.") from last_return_error
        print(
            "Returned working-solution reserve: "
            + ", ".join(
                f"channel {channel + 1} -> {volume_ul:.1f} µL"
                for channel, volume_ul in sorted(returned_by_channel.items())
            )
        )
        
    await discard(WORKING_CHANNELS)
    protocol_state["working_dispensed"] = True
    print("Working solution dispensed to all 30 wells; 500 µL remains in each source tube.")

async def load_assay_plate():
    if not protocol_state["standards_prepared"] or not protocol_state["working_dispensed"]:
        raise RuntimeError("Prepare standards and dispense working solution before loading the remaining plate components.")
    if protocol_state["plate_loaded"]:
        raise RuntimeError("Plate is already marked loaded.")
    for group in PLATE_TRANSFER_GROUPS:
        await dispense_group(group)
        print(f"Completed {group['source']} transfers with {group['tip_size']} µL filtered tips.")
    for well_name in WORKING_WELLS:
        actual = plate_well(well_name).tracker.get_used_volume()
        if abs(actual - 200.0) > 1e-6:
            raise RuntimeError(f"{well_name} volume mismatch: expected 200 µL, tracked {actual} µL.")
    protocol_state["plate_loaded"] = True
    print("Assay plate loading complete: all 30 used wells contain 200 µL.")

def runtime_volume_report():
    report = {key: tube.tracker.get_used_volume() for key, tube in tube_resources.items()}
    if protocol_state["plate_loaded"]:
        for key, expected in PLAN_CHECK["expected_residuals_ul"].items():
            if abs(report[key] - expected) > 1e-6:
                raise RuntimeError(f"{key} residual mismatch: expected {expected} µL, tracked {report[key]} µL.")
    return report




## Run

Uncomment the call below only after checking the deck summary and preflight report. The three stage functions may instead be run individually.


In [10]:
async def run_protocol():
    try:
        # await prepare_standards()
        await dispense_working_solution()
        # await load_assay_plate()
    except Exception:
        if attached_channels:
            print(f"Protocol stopped with tips recorded on physical channels {[c + 1 for c in sorted(attached_channels)]}. Inspect the deck before recovery.")
        raise
    report = runtime_volume_report()
    print("Protocol complete. Tracked source residuals (µL):")
    for source, volume in report.items():
        print(f"  {source:16s} {volume:8.2f}")
    return report

result = await run_protocol()


Working solution batch 1: aspirating 700.0 µL per source at heights [35.09, 35.09] mm.
Working solution batch 2: aspirating 700.0 µL per source at heights [25.24, 25.24] mm.
Protocol stopped with tips recorded on physical channels [5, 6]. Inspect the deck before recovery.


ChannelizedError: ChannelizedError(errors={4: TooLittleVolumeError('Position outside of permitted area'), 5: CommandNotCompletedError('No error')}, raw_response=C0DSid0033er99/00 P502/54 P603/00)

## Recovery

Run only after an interrupted stage reports attached tips and after confirming that movement to the waste is safe. This does not repair liquid volumes or make a partially completed stage safe to rerun.


In [15]:
# await recover_attached_tips()
# await lh.discard_tips()
sources = [tube_resources["working_1"], tube_resources["working_2"]]

await lh.dispense(
    sources,
    vols=[10.0, 10.0],
    use_channels=[4, 5],
    liquid_height=[20.0, 20.0],
    blow_out=[True, True],
)
# await lh.drop_tips(tips_1000["F1:G1"], use_channels=[4, 5])

## Stop

Run only after the protocol or recovery is complete.


In [ ]:
# await lh.stop()
